<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day13_Carnegie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [1]:
url = ("https://raw.githubusercontent.com/ryanaxiom/"
       "Applied-ML/main/data/carnegie_data.csv")
edu_url = ("https://raw.githubusercontent.com/ryanaxiom/"
           "Applied-ML/main/data/education.csv")

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
d["med_yn"] = (d["medical"] == 1).astype(int)
y = d["research"]
pile = ["fallenr20", "totdeg", "facnum", "stem_rsd", "med_yn"]
def rmse(a, b): return np.sqrt(np.mean((a - b)**2))

edu = pd.read_csv(edu_url)
spend = edu["spend"]
three = edu[["income", "young", "urban"]]
d.shape, edu.shape

((302, 103), (50, 6))

In [6]:
rng = np.random.default_rng(309)
idx = rng.choice(len(edu), size = len(edu), replace = True)
print(len(set(idx)))
edu["state"].iloc[idx].value_counts().head(3)

35


,count
state,
MA,3
IA,3
NH,2


In [9]:
rng = np.random.default_rng(309)
coefs = []
for r in range(1000):
    i = rng.choice(len(edu), size=len(edu), replace=True)
    model = LinearRegression()
    model.fit(three.iloc[i], spend.iloc[i])
    coefs.append(model.coef_)
coefs = np.array(coefs)                          # 1000 x 3
for j, name in enumerate(three.columns):
    lo, hi = np.percentile(coefs[:, j], [2.5, 97.5])
    print(name, round(lo, 4), round(hi, 4))

income 0.0334 0.0647
young -0.0169 0.3817
urban -0.0638 0.0044


In [10]:
a, b = edu.index[edu["state"].isin(["AL", "KY"])]
rng = np.random.default_rng(309)
by_absent = {0: [], 1: [], 2: []}
for r in range(1000):
    i = rng.choice(len(edu), size=len(edu), replace=True)
    model = LinearRegression()
    model.fit(three.iloc[i], spend.iloc[i])
    n_absent = int(a not in i) + int(b not in i)
    by_absent[n_absent].append(model.coef_[1])     # young
for k, v in by_absent.items():
    print(k, len(v), round(np.median(v), 3))

0 389 0.032
1 468 0.092
2 143 0.32


In [11]:
rng = np.random.default_rng(309)
bc = []
for r in range(1000):
    i = rng.choice(len(d), size=len(d), replace=True)
    model = LinearRegression()
    model.fit(d[pile].iloc[i], y.iloc[i])
    bc.append(model.coef_)
bc = np.array(bc)
for j, name in enumerate(pile):
    lo, hi = np.percentile(bc[:, j], [2.5, 97.5])
    print(name, round(lo, 1), round(hi, 1))

fallenr20 -17.3 -2.9
totdeg -15.8 39.1
facnum 156.7 323.4
stem_rsd 1241.6 1885.5
med_yn -21914.1 68505.9


In [12]:
two = edu[["income", "young"]]
m2 = LinearRegression()
m2.fit(two, spend)
base2 = rmse(spend, m2.predict(two))
m3 = LinearRegression()
m3.fit(three, spend)
gain = base2 - rmse(spend, m3.predict(three))
print(round(base2, 3), round(gain, 3))

15.419 0.364


In [13]:
rng = np.random.default_rng(309)
fake = []
for r in range(1000):
    sh = three.copy()
    sh["urban"] = rng.permutation(sh["urban"].values)  # shuffle
    ms = LinearRegression()
    ms.fit(sh, spend)
    fake.append(base2 - rmse(spend, ms.predict(sh)))
fake = np.array(fake)
beat = (fake >= gain).sum()
print(beat, round((beat + 1) / 1001, 3))

134 0.135


In [14]:
rng = np.random.default_rng(309)
null = []
for r in range(1000):
    ys = rng.permutation(spend.values)
    ms = LinearRegression()
    ms.fit(three, ys)
    null.append(ms.score(three, ys))
null = np.array(null)
real = m3.score(three, spend)
print(round(real, 3), round(null.max(), 3), (null >= real).sum())

0.472 0.263 0


In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=309)
rng = np.random.default_rng(309)
bump = {c: [] for c in pile}
for tr, te in kf.split(d):
    model = LinearRegression()
    model.fit(d[pile].iloc[tr], y.iloc[tr])
    Xte, yte = d[pile].iloc[te], y.iloc[te]
    base = rmse(yte, model.predict(Xte))
    for c in pile:
        Xs = Xte.copy()
        Xs[c] = rng.permutation(Xs[c].values)     # test rows only
        bump[c].append(rmse(yte, model.predict(Xs)) - base)
for c in pile: print(c, round(np.mean(bump[c])))

fallenr20 80672
totdeg 7265
facnum 155022
stem_rsd 157734
med_yn 307


In [16]:
model = LinearRegression()
rng = np.random.default_rng(309)
leak = []
for r in range(200):
    i = rng.choice(len(edu), size=len(edu), replace=True)
    Xb, yb = three.iloc[i], spend.iloc[i]
    kf_b = KFold(n_splits=5, shuffle=True, random_state=r)
    leak.append(rmse(yb, cross_val_predict(model, Xb, yb, cv=kf_b)))
print(round(np.median(leak), 2), round(min(leak), 2))

16.03 10.42


In [17]:
ames_url = ("https://raw.githubusercontent.com/ryanaxiom/"
            "Applied-ML/main/data/ames_housing.csv")
ames = pd.read_csv(ames_url)
print(ames.isna().sum().sum(), len(ames.dropna()))
ames = pd.read_csv(ames_url, keep_default_na=False,
                   na_values=[""])
print(ames.isna().sum().sum(), len(ames.dropna()))
mv = ames["Mas Vnr Type"]
print(mv.isna().sum(), (mv == "None").sum())

15749 0
719 2269
23 1752
